 ## 셀 1. import

In [1]:
 # 웹캠 탐지/시각화 프로토타입용 기본 import
import cv2
import time
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles

  ## 셀 2. 모델/옵션 설정

In [2]:
# 지금 단계는 분류가 아니라 탐지/전처리 확인 단계입니다.
# 얼굴, 손, 사람 세그멘테이션만 사용합니다.

WINDOW_NAME = "Focus On Class - MediaPipe Prototype"
CAMERA_INDEX = 0
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

BLUE_OVERLAY_ALPHA = 0.35
FACE_BOX_COLOR = (0, 255, 255)
HAND_LABEL_COLOR = (255, 255, 255)
TEXT_COLOR = (255, 255, 255)

FACE_MODEL_PATH = r"mp_model/face_landmarker.task"
HAND_MODEL_PATH = r"mp_model/hand_landmarker.task"
POSE_MODEL_PATH = r"mp_model/pose_landmarker_full.task"

BaseOptions = mp.tasks.BaseOptions
RunningMode = mp.tasks.vision.RunningMode
FaceLandmarker = mp.tasks.vision.FaceLandmarker
HandLandmarker = mp.tasks.vision.HandLandmarker
PoseLandmarker = mp.tasks.vision.PoseLandmarker

face_options = vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=FACE_MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

hand_options = vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

pose_options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=True,
)

  ## 셀 3. 유틸 함수

In [3]:
# 이 셀은 프레임 변환과 그리기 함수를 모아둡니다.
  # 나중에 얼굴 crop 저장기로 확장하기 쉽게 bbox 함수도 따로 둡니다.

def make_mp_image(frame_rgb: np.ndarray) -> mp.Image:
    """RGB 프레임을 MediaPipe 입력 형식으로 바꿉니다."""
    return mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=np.ascontiguousarray(frame_rgb),
    )


def draw_blue_segmentation_overlay(frame_rgb: np.ndarray, pose_result) -> np.ndarray:
    """사람 영역을 파란색 오버레이로 표시합니다."""
    annotated_frame = frame_rgb.copy()

    if not getattr(pose_result, "segmentation_masks", None):
        return annotated_frame

    segmentation_mask = pose_result.segmentation_masks[0].numpy_view()
    segmentation_mask = np.squeeze(segmentation_mask)

    if segmentation_mask.shape[:2] != annotated_frame.shape[:2]:
        segmentation_mask = cv2.resize(
            segmentation_mask,
            (annotated_frame.shape[1], annotated_frame.shape[0]),
            interpolation=cv2.INTER_LINEAR,
        )

    blue_overlay = np.zeros_like(annotated_frame, dtype=np.uint8)
    blue_overlay[:, :, 2] = (segmentation_mask * 255).astype(np.uint8)

    return cv2.addWeighted(
        annotated_frame,
        1.0,
        blue_overlay,
        BLUE_OVERLAY_ALPHA,
        0,
    )


def face_landmarks_to_bbox(face_landmarks, frame_width: int, frame_height: int):
    """얼굴 랜드마크를 bbox로 바꿉니다."""
    xs = [int(lm.x * frame_width) for lm in face_landmarks]
    ys = [int(lm.y * frame_height) for lm in face_landmarks]

    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    pad_x = int((x2 - x1) * 0.12)
    pad_y = int((y2 - y1) * 0.12)

    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(frame_width - 1, x2 + pad_x)
    y2 = min(frame_height - 1, y2 + pad_y)

    return x1, y1, x2, y2


def draw_face_landmarks(frame_rgb: np.ndarray, face_result):
    """얼굴 bbox와 윤곽선을 그립니다."""
    annotated_frame = frame_rgb.copy()
    face_boxes = []

    h, w = annotated_frame.shape[:2]

    for face_idx, face_landmarks in enumerate(getattr(face_result, "face_landmarks", [])):
        x1, y1, x2, y2 = face_landmarks_to_bbox(face_landmarks, w, h)
        face_boxes.append((x1, y1, x2, y2))

        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), FACE_BOX_COLOR, 2)
        cv2.putText(
            annotated_frame,
            f"Face {face_idx + 1}",
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            TEXT_COLOR,
            2,
            cv2.LINE_AA,
        )

        drawing_utils.draw_landmarks(
            image=annotated_frame,
            landmark_list=face_landmarks,
            connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
            landmark_drawing_spec=None,
            connection_drawing_spec=drawing_styles.get_default_face_mesh_contours_style(),
        )

    return annotated_frame, face_boxes


def draw_hand_landmarks(frame_rgb: np.ndarray, hand_result) -> np.ndarray:
    """손 랜드마크와 handedness 라벨을 그립니다."""
    annotated_frame = frame_rgb.copy()

    hand_landmarks_batch = getattr(hand_result, "hand_landmarks", [])
    handedness_batch = getattr(hand_result, "handedness", [])

    for hand_idx, hand_landmarks in enumerate(hand_landmarks_batch):
        drawing_utils.draw_landmarks(
            image=annotated_frame,
            landmark_list=hand_landmarks,
            connections=vision.HandLandmarksConnections.HAND_CONNECTIONS,
            landmark_drawing_spec=drawing_styles.get_default_hand_landmarks_style(),
            connection_drawing_spec=drawing_styles.get_default_hand_connections_style(),
        )

        label = f"Hand {hand_idx + 1}"
        if hand_idx < len(handedness_batch) and handedness_batch[hand_idx]:
            label = f"{handedness_batch[hand_idx][0].category_name} Hand"

        wrist = hand_landmarks[0]
        tx = int(wrist.x * annotated_frame.shape[1])
        ty = int(wrist.y * annotated_frame.shape[0]) - 10

        cv2.putText(
            annotated_frame,
            label,
            (max(0, tx), max(20, ty)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            TEXT_COLOR,
            2,
            cv2.LINE_AA,
        )

    return annotated_frame


def draw_status(frame_rgb: np.ndarray, fps: float, face_count: int, hand_count: int) -> np.ndarray:
    """좌측 상단에 간단한 상태 정보를 표시합니다."""
    annotated_frame = frame_rgb.copy()

    lines = [
        f"FPS: {fps:.1f}",
        f"Faces: {face_count}",
        f"Hands: {hand_count}",
        "Press 'q' to exit",
    ]

    y = 30
    for line in lines:
        cv2.putText(
            annotated_frame,
            line,
            (20, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            TEXT_COLOR,
            2,
            cv2.LINE_AA,
        )
        y += 28

    return annotated_frame

def draw_pose_landmarks(frame_rgb: np.ndarray, pose_result) -> np.ndarray:
      """포즈 랜드마크와 연결선을 화면에 표시합니다."""
      annotated_frame = frame_rgb.copy()

      for pose_landmarks in getattr(pose_result, "pose_landmarks", []):
          drawing_utils.draw_landmarks(
              image=annotated_frame,
              landmark_list=pose_landmarks,
              connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
              landmark_drawing_spec=drawing_styles.get_default_pose_landmarks_style(),
              connection_drawing_spec=drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2),
          )

      return annotated_frame

  ## 셀 4. 메인 실행 루프

In [6]:
 # 웹캠을 열고, 좌우 반전된 프레임에 대해
# 얼굴 탐지 + 손 탐지 + 사람 세그멘테이션을 동시에 보여줍니다.

cv2.destroyAllWindows()

with PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
    FaceLandmarker.create_from_options(face_options) as face_landmarker, \
    HandLandmarker.create_from_options(hand_options) as hand_landmarker:

    camera_capture = cv2.VideoCapture(CAMERA_INDEX)
    camera_capture.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    camera_capture.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

    if not camera_capture.isOpened():
        raise RuntimeError("웹캠을 열 수 없습니다. CAMERA_INDEX 또는 카메라 권한을 확인하세요.")

    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

    start_time = time.perf_counter()
    last_frame_time = time.perf_counter()

    try:
        while True:
            is_readable, frame_bgr = camera_capture.read()
            if not is_readable:
                print("프레임을 읽지 못했습니다.")
                break

            # 거울 모드처럼 보이도록 좌우 반전
            frame_bgr = cv2.flip(frame_bgr, 1)

            # OpenCV(BGR) -> MediaPipe(RGB)
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            mp_image = make_mp_image(frame_rgb)

            # VIDEO 모드용 timestamp
            timestamp_ms = int((time.perf_counter() - start_time) * 1000)

            # 탐지 수행
            pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)
            face_result = face_landmarker.detect_for_video(mp_image, timestamp_ms)
            hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

            annotated_frame_rgb = draw_blue_segmentation_overlay(frame_rgb, pose_result)
            annotated_frame_rgb = draw_pose_landmarks(annotated_frame_rgb, pose_result)
            annotated_frame_rgb, face_boxes = draw_face_landmarks(annotated_frame_rgb, face_result)
            annotated_frame_rgb = draw_hand_landmarks(annotated_frame_rgb, hand_result)

            # FPS 표시
            current_time = time.perf_counter()
            fps = 1.0 / max(1e-6, current_time - last_frame_time)
            last_frame_time = current_time

            face_count = len(face_boxes)
            hand_count = len(getattr(hand_result, "hand_landmarks", []))
            annotated_frame_rgb = draw_status(annotated_frame_rgb, fps, face_count, hand_count)

            # OpenCV 창에는 BGR로 출력
            display_frame_bgr = cv2.cvtColor(annotated_frame_rgb, cv2.COLOR_RGB2BGR)
            cv2.imshow(WINDOW_NAME, display_frame_bgr)

            if (cv2.waitKey(1) & 0xFF) == ord("q"):
                break

    finally:
        camera_capture.release()
        cv2.destroyAllWindows()